# Project Overview


Project Title: Can we predict the game margin between the higher-ranked and lower-ranked player in an ATP match, using pre-match information such as rankings, recent form, surface, tournament context and head-to-head history?

Dataset Name: ATP Tennis 2000 - 2026 Daily update 
Licence: CC0: Public Domain  
Citation: Dissfya. (2026). ATP Tennis 2000 – 2026 Daily update. Kaggle. 
Link: https://www.kaggle.com/datasets/dissfya/atp-tennis-2000-2023daily-pull 


# Imports

In [93]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Data Loading

In [94]:
print(sys.version)
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
pandas 3.1.0 | scikit-learn 1.9.1


In [95]:
seed=42 

In [96]:
raw_data = pd.read_csv("../data/atp_tennis.csv")
raw_data["Date"] = pd.to_datetime(raw_data["Date"])     #change into time 
new_data = raw_data.copy()                                #raw_data stays the same     
print(new_data.shape)
new_data.head()                    #prints first five

(69041, 17)


,Tournament,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
0,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Arthurs W.,Gambill J.M.,Gambill J.M.,105,58,-1,-1,-1.0,-1.0,6-3 6-7 4-6
1,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Balcells J.,Henman T.,Henman T.,218,11,-1,-1,-1.0,-1.0,4-6 6-7
2,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Clement A.,Enqvist T.,Enqvist T.,56,5,-1,-1,-1.0,-1.0,3-6 3-6
3,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Dosedel S.,Ljubicic I.,Dosedel S.,63,77,-1,-1,-1.0,-1.0,6-4 6-2
4,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Draper S.,Norman M.,Norman M.,155,15,-1,-1,-1.0,-1.0,4-6 4-6


# Data Audit

Shape and Type

In [97]:
print(new_data.shape)

(69041, 17)


In [98]:
new_data.info()     #what the variable type is atc

<class 'pandas.DataFrame'>
RangeIndex: 69041 entries, 0 to 69040
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Tournament  69041 non-null  str           
 1   Date        69041 non-null  datetime64[us]
 2   Series      69041 non-null  str           
 3   Court       69041 non-null  str           
 4   Surface     69041 non-null  str           
 5   Round       69041 non-null  str           
 6   Best of     69041 non-null  int64         
 7   Player_1    69041 non-null  str           
 8   Player_2    69041 non-null  str           
 9   Winner      69041 non-null  str           
 10  Rank_1      69041 non-null  int64         
 11  Rank_2      69041 non-null  int64         
 12  Pts_1       69041 non-null  int64         
 13  Pts_2       69041 non-null  int64         
 14  Odd_1       69041 non-null  float64       
 15  Odd_2       69041 non-null  float64       
 16  Score       69041 non-null  str  

The data has 69041 matches(rows) and 17 columns, covering matches from 2000/01/03 to 2026/10/25. `Rank_1`, `Rank_2` and `Best of` are numeric, while Surface, Court, Series and Round are text. `Date` was loaded as text but converted to a datetime so we can do a chronological split on this.

Missing Values

In [99]:
new_data.isna().mean().sort_values(ascending=False)   #see if there is any missing data

Tournament    0.0
Date          0.0
Series        0.0
Court         0.0
Surface       0.0
Round         0.0
Best of       0.0
Player_1      0.0
Player_2      0.0
Winner        0.0
Rank_1        0.0
Rank_2        0.0
Pts_1         0.0
Pts_2         0.0
Odd_1         0.0
Odd_2         0.0
Score         0.0
dtype: float64

In [100]:
new_data[["Rank_1", "Rank_2"]] = new_data[["Rank_1", "Rank_2"]].mask(new_data[["Rank_1", "Rank_2"]] <= 0)


print(new_data[["Rank_1", "Rank_2"]].isna().sum())                       # number of ranks that are less than 0 (this means they are unranked players)

Rank_1    14
Rank_2    12
dtype: int64


We can see there are no missing values throughout our dataset. However we can see that the number of players with a rank less than zero is 26. These are unranked players and have a rank of -1. These are removed from the data as a rank of -1 would be read as the best possible rank, so the model would treat unranked players as the top seeds.

In [101]:
n = len(new_data)
unranked = (new_data["Rank_1"].isna() | new_data["Rank_2"].isna()
            | (new_data["Rank_1"] <= 0) | (new_data["Rank_2"] <= 0))
print("Removed matches with an unranked player:", unranked.sum())
new_data = new_data[~unranked]

Removed matches with an unranked player: 26


Duplicates

In [102]:
key =["Date", "Tournament", "Player_1", "Player_2"]              #if all these are the same then match is a duplicate
print("Exact duplicate rows:", new_data.duplicated().sum())              #actual same rows 
print("Same match (by key):", new_data.duplicated(subset=key).sum())        #using key

Exact duplicate rows: 0
Same match (by key): 2


There are 0 exact duplicate rows however we can there are 2 matches that are the same by using our key ("Date", "Tournament", "Player_1", "Player_2"). These matches are removed from the data as a match counted twice gets double weight in training.

In [104]:
new_data=new_data.drop_duplicates(subset=key)            #remove these rows 

Coverage, surface and incomplete matches

In [105]:
print(new_data["Surface"].value_counts())                      # shows the surfaces and that hard court dominates

Surface
Hard      37508
Clay      22144
Grass      7729
Carpet     1632
Name: count, dtype: int64


In [106]:
print(new_data["Date"].dt.year.value_counts().sort_index())    #matches per year (highlightes significantly less during covid (2020)

Date
2000    2864
2001    2968
2002    2694
2003    2697
2004    2782
2005    2815
2006    2793
2007    2712
2008    2565
2009    2607
2010    2562
2011    2565
2012    2508
2013    2538
2014    2448
2015    2518
2016    2520
2017    2528
2018    2568
2019    2497
2020    1231
2021    2402
2022    2545
2023    2607
2024    2631
2025    2487
2026    2361
Name: count, dtype: int64


In [107]:
print("Incomplete scores:", new_data["Score"].str.contains("RET|W/O|DEF", case=False, na=False).sum())    #Number of incomplete scores(walkovers or retirements etc)


Incomplete scores: 0


We can see there are signficantly less matches during 2020 as a result of Covid 19. Surface split is 54.33% hard court, 32.07% clay, 11.19% grass and 2.36% carpet so hard court dominamtes as expected. There was 0 incomplete matches which includes


In [51]:
new_data[["Rank_1", "Rank_2"]].describe()                      # rank outliers(however not fully accurate yet as unranked players are still included with a rank of -1)

,Rank_1,Rank_2
count,69039.000000,69039.000000
mean,75.861151,75.416243
std,100.393808,100.668209
min,-1.000000,-1.000000
25%,25.000000,24.000000
50%,54.000000,54.000000
75%,92.000000,92.000000
max,3390.000000,4915.000000


# build the target

In [56]:
# remove tied rankings
tied = new_data["Rank_1"] == new_data["Rank_2"]
print("Removed tied ranks:", tied.sum())
new_data = new_data[~tied]

Removed tied ranks: 0


In [ ]:
#no tied rankings to remove anyway

In [ ]:
#check and remove any matches where winnner doesnt equal any of the two players that played

In [81]:
bad_winner = ~((new_data["Winner"] == new_data["Player_1"]) | (new_data["Winner"] == new_data["Player_2"]))
print("Winner matches neither player:", bad_winner.sum())
new_data = new_data[~bad_winner].copy()

Winner matches neither player: 0


In [62]:
#none anyway

In [ ]:
#change the scores of the matches into number of games won by both players